In [13]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq

load_dotenv()
groq_key = os.getenv("GROQ_API_KEY")
tavily_key = os.getenv("TAVILY_API_KEY")

llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)
print(" Setup complete. LangSmith tracing:", os.getenv("LANGSMITH_TRACING"))

 Setup complete. LangSmith tracing: true


In [14]:
from langchain_core.tools import tool
from tavily import TavilyClient
import time

tavily_client = TavilyClient(api_key=tavily_key)

@tool
def web_search(query: str) -> str:
    """Search the web for current information on a topic. Use this when you need
    up-to-date facts, news, or information that might not be in your training data."""
    max_retries = 3
    for attempt in range(max_retries):
        try:
            results = tavily_client.search(query=query, max_results=3)
            formatted = "\n\n".join(
                f"Title: {r['title']}\nURL: {r['url']}\nContent: {r['content'][:300]}"
                for r in results["results"]
            )
            return formatted
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f" Search failed ({e.__class__.__name__}), retrying in {wait}s...")
                time.sleep(wait)
            else:
                return f"Search failed after {max_retries} attempts: {e}"

print(" Tool defined with retry logic:", web_search.name)

 Tool defined with retry logic: web_search


In [15]:
llm_with_tools = llm.bind_tools([web_search])
print("LLM now knows about the web_search tool")

LLM now knows about the web_search tool


In [16]:
response = llm_with_tools.invoke("What's the latest news on solid-state EV battery safety recalls in 2026?")

print("Model's text response:", response.content)
print("\nDid it decide to call a tool?", bool(response.tool_calls))
if response.tool_calls:
    for tc in response.tool_calls:
        print("Tool requested:", tc["name"])
        print("Arguments:", tc["args"])

Model's text response: I need to clarify an important point: **2026 has not yet occurred** (as of my current knowledge cutoff and the present date). Therefore, there is no "latest news" from 2026, as that year is in the future.

However, I can provide you with the **most recent available information** on solid-state EV battery safety recalls and developments up to the current date. Let me search for the latest news on this topic.



Did it decide to call a tool? True
Tool requested: web_search
Arguments: {'query': 'solid-state EV battery safety recalls 2024 2025 latest news'}


In [17]:
response2 = llm_with_tools.invoke("What is 15% of 240?")

print("Model's text response:", response2.content)
print("Did it decide to call a tool?", bool(response2.tool_calls))

Model's text response: To find 15% of 240, you can multiply 240 by 0.15:

$$ 240 \times 0.15 = 36 $$

Alternatively, you can break it down:
*   10% of 240 is 24.
*   5% of 240 is 12.
*   $24 + 12 = 36$.

**Answer: 36**
Did it decide to call a tool? False


In [18]:
from langchain_core.messages import HumanMessage, ToolMessage

messages = [HumanMessage(content="What's the latest news on solid-state EV battery safety recalls in 2026?")]
ai_response = llm_with_tools.invoke(messages)
messages.append(ai_response)

if ai_response.tool_calls:
    for tool_call in ai_response.tool_calls:
        print(f"🔧 Executing: {tool_call['name']}({tool_call['args']})")
        tool_result = web_search.invoke(tool_call["args"])
        messages.append(ToolMessage(content=tool_result, tool_call_id=tool_call["id"]))

    final_response = llm_with_tools.invoke(messages)
    print("\n--- Final answer, grounded in real search results ---")
    print(final_response.content)

🔧 Executing: web_search({'query': 'solid-state EV battery safety recalls 2024 2025 latest news'})

--- Final answer, grounded in real search results ---



In [21]:
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, MessagesState, START, END

tool_node = ToolNode([web_search])

from langchain_core.messages import SystemMessage

def agent_node(state: MessagesState) -> dict:
    messages = state["messages"]
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content="Always provide a complete, substantive text answer to the user based on any tool results. Never return an empty response.")] + messages
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

def agent_node(state: MessagesState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: MessagesState) -> str:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END

graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")  # after tool runs, go back to the agent to form a final answer

app = graph.compile()
print("Tool-calling graph compiled")

Tool-calling graph compiled


In [20]:
result = app.invoke({"messages": [HumanMessage(content="What's the latest news on solid-state EV battery safety recalls in 2026?")]})

for msg in result["messages"]:
    print(f"[{msg.type}] {msg.content[:200] if msg.content else '(tool call)'}")

[human] What's the latest news on solid-state EV battery safety recalls in 2026?
[ai] I need to clarify an important point: **2026 has not yet occurred** (as of my current knowledge cutoff and the present date). Therefore, there is no "latest news" from 2026, as that year is in the fut
[tool] Title: Solid-State Batteries EXPOSED: The 800-Mile EV Claim ...
URL: https://www.youtube.com/watch?v=c8aHDqWgeuU&vl=en
Content: scale before, so their conservative tone may be the most credible clue a
[ai] (tool call)
[tool] Title: Solid-state batteries set to redefine EV performance, safety
URL: https://www.cbtnews.com/solid-state-batteries-set-to-redefine-ev-performance-safety
Content: TSLA377.940-2.18%

GM80.570-3.2%


[ai] (tool call)
[tool] Title: Part 573 Safety Recall Report 25V-185
URL: https://static.nhtsa.gov/odi/rcl/2025/RCLRPT-25V185-3702.PDF
Content: OMB Control No.: 2127-0004 Part 573 Safety Recall Report 25V-185 The information
[ai] Based on my research, here's what I can tell you:
